# Level 4 — 심층 분석: Grad-CAM 및 효율성 Trade-off

**목표**: 모델의 *동작 원리* 를 설명하고, FPS와 정확도의 trade-off 를 정량화합니다.

리포트 필수 산출물:
1. **속성별 정규화 Confusion Matrix 3개** — best 모델 기준.
2. **Grad-CAM 패널** — 같은 이미지에 대해 3개 head 가 각각 어디를 보는지 시각화 (예: `rainy + night + city street` 인 이미지).
3. 모든 백본에 대한 **FPS vs Avg-MF1 Pareto plot**.

본 노트북은 학습 노트북이 아니라 분석 노트북이지만, wandb 가 활성화되어 있으면 confusion matrix 이미지·Grad-CAM 패널·FPS 표를 같은 프로젝트의 별도 Run 으로 업로드합니다.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/Lee-Yongsu/2026-HYU-AUE8088-PA2.git
else:
    !git pull origin main

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed
from src.utils.transforms import eval_transform
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, average_macro_f1, CLASS_NAMES
from src.utils.efficiency import measure_fps
from src.utils.wandb_logger import WandbLogger
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.xai.gradcam import GradCAM
from src.models.resnet import resnet18, resnet50
from src.models.vit import vit_small_patch16_224

set_seed(42, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
logger = WandbLogger(project=WANDB_PROJECT, run_name="level4-analysis", tags=["level4", "analysis"])

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# 분석 대상으로 사용할 best 모델 로드 (Level 3 best: ViT-S/16 pretrained + loss+sampler+mix)
val_ds = BDDAttrDataset("../data/set_a", "val", transform=eval_transform())
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

# 체크포인트 경로 — 환경마다 위치가 달라 후보를 순서대로 탐색
#   ../checkpoints  : Colab 관례 (repo 상위에 저장)
#   checkpoints     : repo 내부 (로컬에 직접 받아둔 경우)
_CKPT_CANDIDATES = [
    "../checkpoints/level3_loss_sampler_mix.pth",
    "checkpoints/level3_loss_sampler_mix.pth",
]
BEST_CKPT = next((p for p in _CKPT_CANDIDATES if os.path.exists(p)), _CKPT_CANDIDATES[0])
print("loading checkpoint:", BEST_CKPT)

model = vit_small_patch16_224().to(device)
state = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(state)   # Level 2/3 형식 = raw state_dict (["state_dict"] 래퍼 없음)
model.eval()

In [ ]:
# 속성별 정규화 Confusion Matrix 생성 및 시각화
preds, probs, targets, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(preds, targets, normalize="true")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", ax=ax, cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a])
    ax.set_title(f"{a}")
    ax.set_xlabel("예측 (pred)"); ax.set_ylabel("정답 (true)")
fig.tight_layout()
logger.log_image("analysis/confusion_matrices", fig)

# 속성별 개별 confusion matrix 도 분리해서 업로드
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"analysis/cm_{a}", cms[a], CLASS_NAMES[a])

In [ ]:
# 속성별 Grad-CAM. ViT 는 토큰 시퀀스(B,1+N,D) 라 14x14 패치 그리드로 reshape 후 CAM 계산.
import torch.nn.functional as F

class ViTGradCAM(GradCAM):
    """ViT 토큰을 패치 그리드로 reshape 하여 표준 Grad-CAM 을 적용 (cls 토큰 제외)."""
    def __init__(self, model, target_layer, grid=14):
        super().__init__(model, target_layer)
        self.grid = grid
    def _to_grid(self, t):                        # (B,1+N,D) -> (B,D,H,W)
        B, _, D = t.shape
        return t[:, 1:, :].reshape(B, self.grid, self.grid, D).permute(0, 3, 1, 2)
    def __call__(self, x, score_fn):
        self.model.zero_grad()
        score_fn(self.model(x)).backward(retain_graph=True)
        a, g = self._to_grid(self._activations), self._to_grid(self._gradients)
        weights = g.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * a).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam_min = cam.amin(dim=(2, 3), keepdim=True); cam_max = cam.amax(dim=(2, 3), keepdim=True)
        return ((cam - cam_min) / (cam_max - cam_min + 1e-8)).squeeze(1)

# patch 정보가 cls 토큰으로 섞이기 직전인 마지막 블록의 norm1 을 target 으로 사용.
gc = ViTGradCAM(model, model.blocks[-1].norm1, grid=14)

batch = next(iter(val_loader))
x = batch["image"][:6].to(device)  # 샘플 이미지 6장

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row, attr in enumerate(ATTRIBUTES):
    # 각 속성 head 의 최대 logit 합을 score 로 사용 → 해당 head 가 "보는" 영역 추출
    cam = gc(x, lambda out, a=attr: out[a].max(dim=-1).values.sum())
    for col in range(6):
        img = x[col].cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())
        axes[row, col].imshow(img)
        axes[row, col].imshow(cam[col].cpu().numpy(), cmap="jet", alpha=0.45)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(attr, fontsize=14)
fig.tight_layout()
logger.log_image("analysis/gradcam_panel", fig)

In [ ]:
# FPS 측정 — 배치 1, 224x224, warm-up 20회 후 200회 평균. 모든 백본에 대해 실행.
fps_rows = []
fps_map = {}
for fn, name in [(resnet18, "resnet18"), (resnet50, "resnet50"), (vit_small_patch16_224, "vit_s16")]:
    m = fn().to(device).eval()
    fps = measure_fps(m, device, batch_size=1, n_warmup=20, n_iter=200)
    print(f"{name:10s} FPS = {fps:.2f}")
    fps_rows.append([name, round(fps, 2)])
    fps_map[name] = fps

logger.log_table("analysis/fps", ["backbone", "FPS"], fps_rows)

# FPS (x축) vs Avg-MF1 (y축) Pareto plot.
# Avg-MF1 은 각 모델 학습 때 wandb 에 기록된 최종값을 입력 (체크포인트 재평가 불필요).
avg_mf1 = {
    "resnet18": 0.00,   # TODO: Level1 기록값으로 채우기
    "resnet50": 0.00,   # TODO: Level1 기록값으로 채우기
    "vit_s16":  0.00,   # TODO: best run (level3 loss+sampler+mix) Avg-MF1 로 채우기
}

pts = [(fps_map[n], avg_mf1[n], n) for n in fps_map]
def _dominated(p):
    # 다른 점 q 가 FPS·Avg-MF1 둘 다 우월하면 p 는 지배됨 (Pareto 최적 아님)
    return any(q != p and q[0] >= p[0] and q[1] >= p[1] and (q[0] > p[0] or q[1] > p[1]) for q in pts)
front = sorted(p for p in pts if not _dominated(p))

fig, ax = plt.subplots(figsize=(6, 5))
for f, mf1, n in pts:
    ax.scatter(f, mf1, s=80)
    ax.annotate(n, (f, mf1), textcoords="offset points", xytext=(5, 5))
ax.plot([p[0] for p in front], [p[1] for p in front], "r--", label="Pareto front")
ax.set_xlabel("FPS (T4, batch=1)"); ax.set_ylabel("Avg-Macro-F1"); ax.legend()
fig.tight_layout()
logger.log_image("analysis/pareto", fig)

In [ ]:
logger.finish()